# Multi-GPU sanity check: 10 jobs on T4 x2

A short, one-time check before the real Phase 0 campaign: does
`multi_worker_phase0.run_multi_gpu_chunk` (README's "2-GPU support" section) actually
use **both** GPUs on real Kaggle hardware, and do the two workers' results merge
correctly? Should take well under 30 minutes for 10 jobs (8 logging + 1 ground-truth
fold pair = 10 jobs, split across 2 workers, ~5 each).

**No Kaggle Dataset checkpoint round-trip here** -- unlike `orchestrated_pilot.ipynb`,
this is a one-off local check, not a resumable session, so no `KAGGLE_API_TOKEN`
secret or `DATASET_SLUG` is needed.

**Update (2026-09-30)**: the first real run of this notebook hung (both workers loading the model at once). Fixed at the source -- pre-download once, staggered loading, and a 20-minute watchdog that kills both workers and fails loudly instead of hanging. See README's "Multi-GPU sanity check froze on Kaggle" section for the full diagnosis. No changes needed in this notebook itself.

## Settings to click

- Accelerator: **GPU T4 x2** (this is the whole point of the check).
- Internet: **ON** (for pip installs and downloads).

## What this checks

1. Both GPUs actually got used (each worker's results file is tagged with its own
   `worker_id`, and each worker prints its GPU's `nvidia-smi` view before starting).
2. Both `ground_truth_fold_a`/`_b` arms of the one fold pair landed on the SAME
   worker (never split across GPUs).
3. `merge_worker_results` produces exactly 10 unique episodes, one per job_id, no
   duplicates and nothing missing.

## 0. Setup

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import subprocess
print('GPUs visible to this notebook process:')
subprocess.run(['nvidia-smi', '-L'])


In [ ]:
!git clone https://github.com/ibo-agentic/memory-pilot.git
%cd memory-pilot
!git rev-parse HEAD


In [ ]:
!pip install alfworld tqdm termcolor sentence-transformers --quiet
!pip install transformers accelerate bitsandbytes --quiet
!alfworld-download
!pip install -e . --quiet
!pip install -e alfworld_pilot --quiet


In [ ]:
import sys, site, pathlib, importlib

importlib.reload(site)
site_dirs = list(site.getsitepackages())
try:
    site_dirs.append(site.getusersitepackages())
except Exception:
    pass
for _dir in site_dirs:
    if _dir and pathlib.Path(_dir).is_dir():
        site.addsitedir(_dir)

for _mod_name in list(sys.modules):
    if _mod_name == 'alfworld_pilot' or _mod_name.startswith('alfworld_pilot.') or _mod_name == 'memory_ope' or _mod_name.startswith('memory_ope.'):
        del sys.modules[_mod_name]

print('sys.path refreshed and cached alfworld_pilot/memory_ope modules cleared.')


In [ ]:
try:
    import alfworld_pilot
    import memory_ope
    from alfworld_pilot import job_list, multi_worker_phase0, run_jobs  # noqa: F401
    print('Import check PASSED:')
    print('  alfworld_pilot ->', alfworld_pilot.__file__)
    print('  memory_ope     ->', memory_ope.__file__)
except ImportError as e:
    raise SystemExit(f'Import check FAILED: {e!r} -- fix the install cell above before proceeding.')


## 1. Generate a tiny job list (10 jobs) and run it across both GPUs

In [ ]:
%cd alfworld_pilot
import time, pathlib
from alfworld_pilot.job_list import generate_job_list, write_job_list
from alfworld_pilot.multi_worker_phase0 import run_multi_gpu_chunk, merge_worker_results

SANITY_DIR = pathlib.Path('/kaggle/working/multi_gpu_sanity')
SANITY_DIR.mkdir(parents=True, exist_ok=True)

jobs = generate_job_list(n_logging=8, n_gt_pairs=1, logging_start_seed=5000, gt_start_seed=1_500_000)
job_list_path = SANITY_DIR / 'jobs.json'
write_job_list(jobs, job_list_path)
print(f'Generated {len(jobs)} jobs (8 logging + 1 ground-truth fold pair = 2 jobs).')

worker_results_paths = [SANITY_DIR / f'worker{i}_results.jsonl' for i in range(2)]

t0 = time.monotonic()
n_new = run_multi_gpu_chunk(
    job_list_path, worker_results_paths, chunk_budget_seconds=25 * 60, work_dir=SANITY_DIR / 'chunks', backend='real',
)
elapsed = time.monotonic() - t0
print(f'Completed {n_new} job(s) in {elapsed:.0f}s ({elapsed / 60:.1f} min).')


## 2. Check both GPUs were actually used

In [ ]:
import json

for i, path in enumerate(worker_results_paths):
    if not path.exists():
        print(f'Worker {i}: NO results file at all -- did not run or crashed before completing anything.')
        continue
    with open(path) as f:
        rows = [json.loads(line) for line in f if line.strip()]
    worker_ids_seen = {r['worker_id'] for r in rows}
    print(f'Worker {i} results file: {len(rows)} episode(s), worker_id tag(s) seen: {worker_ids_seen}')

if all(p.exists() for p in worker_results_paths):
    print()
    print('PASS' if all(p.stat().st_size > 0 for p in worker_results_paths) else 'FAIL',
          ': both workers wrote at least one episode -- both GPUs were used.' if all(p.stat().st_size > 0 for p in worker_results_paths)
          else ': at least one worker produced nothing.')


## 3. Check the fold pair stayed on one worker, and the merge is correct

In [ ]:
fold_worker_ids = {}
for i, path in enumerate(worker_results_paths):
    if not path.exists():
        continue
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            if 'fold_over_pair_index' in r:
                fold_worker_ids.setdefault(r['fold_over_pair_index'], set()).add(i)

for pair_index, workers in fold_worker_ids.items():
    status = 'PASS' if len(workers) == 1 else 'FAIL'
    print(f'{status}: fold pair {pair_index} ran on worker(s) {workers}' + (' (both arms together, as required)' if len(workers) == 1 else ' -- SPLIT ACROSS WORKERS, this should never happen'))

merged_path = SANITY_DIR / 'merged.jsonl'
n_written = merge_worker_results(worker_results_paths, merged_path)
with open(merged_path) as f:
    merged_ids = [json.loads(line)['job_id'] for line in f if line.strip()]

expected_ids = {j['job_id'] for j in jobs}
print()
print(f'Merged: {n_written} episode(s), {len(set(merged_ids))} unique job_id(s) (expected {len(expected_ids)}).')
print('PASS' if set(merged_ids) == expected_ids and len(merged_ids) == len(set(merged_ids)) else 'FAIL',
      ': merged file exactly matches the job list, no duplicates, nothing missing.')


## Done

Report back: whether both PASS lines above printed PASS, and the total elapsed time
from Section 1 (should be well under 30 minutes). If anything printed FAIL, paste it
back before launching the real Phase 0 campaign in `orchestrated_pilot.ipynb`.